In [21]:
import json
import unicodedata
import urllib.request

URL = "https://raw.githubusercontent.com/words/an-array-of-french-words/master/index.json"

def sans_diacritiques(mot: str) -> str:
    nkd = unicodedata.normalize("NFD", mot)
    return "".join(c for c in nkd if not unicodedata.combining(c))

# Télécharger
with urllib.request.urlopen(URL) as resp:
    mots_bruts = json.load(resp)

# Nettoyer : minuscules, sans accents, uniques, lettres seulement
mots = sorted({
    sans_diacritiques(m.lower())
    for m in mots_bruts
    if m.isalpha()
})

print(f"{len(mots)} mots chargés")

# Sauvegarder en local (sert de cache pour les prochains runs)
with open("mots_fr.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(mots))

def load_words_from_file(filename: str) -> list[str]:
    with open(filename, "r", encoding="utf-8") as f:
        return [line.strip() for line in f.readlines()]

318883 mots chargés


In [22]:
import unicodedata
import pandas as pd

LEXIQUE_URL = "http://www.lexique.org/databases/Lexique383/Lexique383.tsv"
lex = pd.read_csv(LEXIQUE_URL, sep="\t", low_memory=False)
lex = lex.dropna(subset=["ortho"])

def sans_diacritiques(mot) -> str:
    if not isinstance(mot, str):
        return ""
    nkd = unicodedata.normalize("NFD", mot)
    return "".join(c for c in nkd if not unicodedata.combining(c))

lex["cgram"] = lex["cgram"].fillna("")
lex["infover"] = lex["infover"].fillna("")
lex["lemme"] = lex["lemme"].fillna("")

garde = set()        # au moins une lecture NON-conjugaison
a_une_conjugaison = set()

for row in lex.itertuples(index=False):
    cle = sans_diacritiques(row.ortho.lower())
    if not cle or not cle.isalpha():
        continue

    if row.cgram != "VER":
        garde.add(cle)          # nom, adjectif, etc. → protège toujours
        continue

    if row.lemme == row.ortho:
        garde.add(cle)          # infinitif
    elif ":" in row.infover:
        a_une_conjugaison.add(cle)   # ex. ";ind:prés:3s" → vraie conjugaison
    else:
        garde.add(cle)          # participe (code sans ":")

# ne retirer que si TOUTES les lectures sont des conjugaisons
a_retirer = a_une_conjugaison - garde

print(f"{len(a_retirer)} formes purement conjuguées à retirer")

46059 formes purement conjuguées à retirer


In [23]:
with open("mots_fr.txt", encoding="utf-8") as f:
    mots = f.read().splitlines()

mots_filtres = [m for m in mots if m not in a_retirer]
print(f"{len(mots)} -> {len(mots_filtres)} ({len(mots)-len(mots_filtres)} retirés)")

for m in ("mangeaient", "peuvent", "manger", "ete", "portant", "vent", "as"):
    print(f"{m:12} {'OK' if m in mots_filtres else 'retiré'}")

# Saving filtered words to a new file
with open("mots_fr_filtres.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(mots_filtres))

318883 -> 274268 (44615 retirés)
mangeaient   retiré
peuvent      retiré
manger       OK
ete          OK
portant      OK
vent         OK
as           OK


In [24]:
encoding_dict = {a: i for i, a in enumerate("abcdefghijklmnopqrstuvwxyz")}
decoding_dict = {i: a for i, a in enumerate("abcdefghijklmnopqrstuvwxyz")}

In [25]:
import numpy as np

def extract_words_len_k(mots,k):
    return [m for m in mots if len(m) == k]
def extract_words_starting_with(mots, start):
    return [m for m in mots if m.startswith(start)]

def compute_letter_probs(mots):
    letter_counts = [0 for i in range(26)]
    double_letter_counts = [0 for i in range(26)]
    total_letters = 0
    for m in mots:
        double_letter_seen = set()
        for c in m:
            letter_counts[encoding_dict[c]] += 1
            total_letters += 1
            if c in double_letter_seen:
                double_letter_counts[encoding_dict[c]] += 1
            double_letter_seen.add(c)
    probs1 = [count / total_letters if total_letters > 0 else 0 for count in letter_counts]
    probs2 = [count / total_letters if total_letters > 0 else 0 for count in double_letter_counts]
    return probs1, probs2

def compute_letter_probs_with_position(mots):
    mots_probs_1 = np.zeros((26, len(mots[0])))
    mots_probs_2 = np.zeros((26, len(mots[0])))
    total_letters = 0
    for m in mots:
        double_letter_seen = set()
        for i, c in enumerate(m):
            mots_probs_1[encoding_dict[c], i] += 1
            total_letters += 1
            if c in double_letter_seen:
                mots_probs_2[encoding_dict[c], i] += 1
            double_letter_seen.add(c)
    mots_probs_1 /= total_letters
    mots_probs_2 /= total_letters
    return mots_probs_1, mots_probs_2

In [ ]:
import math
def compute_words_with_most_information(mots,prob_letters, prob_double_letters, mots_possibles):
    # mots of len k starting with all the same letter
    # Doing it with backtracking using the fact that mots is sorted lexicographically
    # the information of a word is the sum of the information of its letters, which is -log(prob(letter)) 
    # Question : Then I should pick the lowest value of sums of log probs as best word ? ANSWER : Yes, because the information content is the negative log of the probability, so a lower probability means a higher information content.
    # But if the probability is lower their log is negative and lower, so the -log is higher for less probable letters, which is what we want. So we want to pick the word with the highest sum of -log(prob(letter)) values.
    # But why because higher -log means less probable letters, and we want the word with the most probable letters to give us info.
    # Hence, we want to pick the word with the highest information content, thus 
    # Also for letters that happen more than once, we for the second time use the probabilty of double letter, and for the 3rd time no prob 
    # (prob of 3 letters is set to 0, but in log we"ll put it to -100, as if all words have 3 letters we still want to pick the best
    
    best_word = ""
    best_info = -1000000
    for word in mots:
        info = 0
        letter_seen = {}
        for c in word:
            if c not in letter_seen:
                info += prob_letters[encoding_dict[c]]
                letter_seen[c] = 1
            elif letter_seen[c] == 1:
                info += prob_double_letters[encoding_dict[c]]
                letter_seen[c] += 1
        if word in mots_possibles:
            info *= 1.2
        if info > best_info:
            best_info = info
            best_word = word 
    return best_word
def compute_words_with_most_information_with_position(mots,prob_letters, prob_double_letters,mots_possibles):
    best_word = ""
    best_info = -1000000
    probs_letters_raw = np.sum(prob_letters,axis=1)
    probs_double_letters_raw = np.sum(prob_double_letters,axis=1)
    candidats = set(mots_possibles)
    for word in mots:
        info = 0
        letter_seen = {}
        for i, c in enumerate(word):
            if c not in letter_seen:
                info += prob_letters[encoding_dict[c], i] + 3/4*probs_letters_raw[encoding_dict[c]]
                letter_seen[c] = 1
            elif letter_seen[c] == 1:
                info += prob_double_letters[encoding_dict[c], i] + 3/4*probs_double_letters_raw[encoding_dict[c]]
                letter_seen[c] += 1
        if word in candidats:
            info *= 1.2
        if info > best_info:
            best_info = info
            best_word = word 
    return best_word

In [28]:
# test case for len 8 words with starting letter o
mots_test = extract_words_len_k(extract_words_starting_with(mots, "r"), 12)
probs1, probs2 = compute_letter_probs(mots_test)
print(f"probs1: {probs1}")
best_word = compute_words_with_most_information(mots_test,probs1,probs2,mots_test)
probs_with_position1, probs_with_position2 = compute_letter_probs_with_position(mots_test)
best_word_with_position = compute_words_with_most_information_with_position(mots_test,probs_with_position1,probs_with_position2)
print(f"best_word: {best_word} and best_word_with_position: {best_word_with_position}")

probs1: [0.08880090497737557, 0.009539969834087482, 0.029468325791855203, 0.01349924585218703, 0.16989064856711916, 0.009709653092006033, 0.013254147812971342, 0.007541478129713424, 0.08904600301659125, 0.0011689291101055807, 3.770739064856712e-05, 0.027639517345399697, 0.022737556561085972, 0.07141779788838612, 0.053638763197586724, 0.019193061840120665, 0.0038273001508295624, 0.1553921568627451, 0.10273378582202111, 0.06197209653092006, 0.026564856711915535, 0.008974358974358974, 0.0, 0.0005844645550527903, 0.0014328808446455504, 0.011934389140271492]
best_word: recroissante and best_word_with_position: relaisseront


In [29]:
def merge_two_sorted_lists(l1,l2):
    res = []
    a = l1.pop()
    b = l2.pop()
    while l1!= [] and l2!=[]: 
        if a<b:
            res.append(b)
            b = l2.pop()
        else:
            res.append(a)
            a = l1.pop()
    if a<b:
        res.append(b)
        res.append(a)
    else:
        res.append(a)
        res.append(b)

    res = l1 + l2 + res
    ret = [res[-i] for i in range(len(res))]
    return ret

def less_than_k_letter_c(c,k,word):
    j = 0
    for s in word:
        j +=(s==c)
        if j>k:
            return False
    return True


In [82]:
# We now consider that we know a partial information on the word : some letter's place, or that they exist somewhere in the word an we use this for our next guess
def update_probs_result(mots_possibles,result,tested_word,fixed_letters):
    for i,c in enumerate(tested_word):
        if result[i] == 0:
            if c in fixed_letters or c in [tested_word[i] for i in range(len(tested_word)) if result[i]>0]:
                j = 0
                for s in fixed_letters:
                    j +=(s==c)
                for k,letter in enumerate(tested_word):
                    if letter==c and result[k]>=1:
                        j += 1
                mots_possibles = [m for m in mots_possibles if less_than_k_letter_c(c,j,m)]


            else:
                mots_possibles = [m for m in mots_possibles if c not in m]
        elif result[i] == 1:
            mots_possibles = [m for m in mots_possibles if m[i] == c]
            fixed_letters[i] = c
        elif result[i] == 2:
            mots_possibles = [m for m in mots_possibles if c in m and m[i] != c]
    mots_probs_1 = np.zeros((26, len(mots_possibles[0])))
    mots_probs_2 = np.zeros((26, len(mots_possibles[0])))
    total_letters = 0
    for m in mots_possibles:
        double_letter_seen = set()
        for i, c in enumerate(m):
            mots_probs_1[encoding_dict[c], i] += 1
            total_letters += 1
            if c in double_letter_seen:
                mots_probs_2[encoding_dict[c], i] += 1
            double_letter_seen.add(c)
    mots_probs_1 /= total_letters
    mots_probs_2 /= total_letters
    return mots_possibles, mots_probs_1, mots_probs_2,fixed_letters

def game_loop():
    starting_letter = input("Enter the starting letter: ")
    word_length = int(input("Enter the word length: "))

    mots = load_words_from_file("mots_fr.txt")
    mots = extract_words_len_k(extract_words_starting_with(mots, starting_letter), word_length)
    mots_possibles = load_words_from_file("mots_fr_filtres.txt")
    mots_possibles = extract_words_len_k(extract_words_starting_with(mots_possibles, starting_letter), word_length)
    possible_letters = []
    fixed_letters = ["."]*word_length
    probs_with_position1, probs_with_position2 = compute_letter_probs_with_position(mots)
    while len(mots_possibles) > 1:
        best_word_with_position = compute_words_with_most_information_with_position(mots,probs_with_position1,probs_with_position2,mots_possibles)
        print(f"Best guess: {best_word_with_position}")
        result_str = input("Enter the result (0 for not in word, 2 for wrong position, 1 for correct position): ")
        result = [int(x) for x in result_str]
        mots_possibles, probs_with_position1, probs_with_position2,fixed_letters = update_probs_result(mots_possibles, result, best_word_with_position,fixed_letters)
        print(f"{len(mots_possibles)} words remaining.")
        if len(mots_possibles)<= 10:
            print(f"Remaining words: {mots_possibles}")

game_loop()   
    

Best guess: dentaires
20 words remaining.
Best guess: depilames
2 words remaining.
Remaining words: ['defailles', 'delaisses']
Best guess: defailles


IndexError: list index out of range